### libraries

In [16]:
import requests
import numpy as np
import fastf1
from pathlib import Path
import pandas as pd

### load dataframes

In [17]:
working_directory = Path.cwd().parent.parent
print(working_directory)

file_path_1 = working_directory / "data" / "clean" / "driver_points.csv"
file_path_2 = working_directory / "data" / "clean" / "driver_price.csv"
file_path_3 = working_directory / "data" / "clean" / "driver_roster.csv"



/Users/jackguptill/Library/CloudStorage/OneDrive-Personal/Code/F1FantasyProject


In [18]:
driver_points = pd.read_csv(file_path_1).drop(columns=["Unnamed: 0"])
driver_price = pd.read_csv(file_path_2).drop(columns=["Unnamed: 0"])
driver_roster = pd.read_csv(file_path_3).drop(columns=["Unnamed: 0"])

In [19]:
driver_points.tail()

,driver,race,points,year,race_name,start_date,location,country_code,month,start_epoch
1627,BOR,2,NaN,2026,Chinese Grand Prix,2026-03-15 07:00:00+00:00,Shanghai,CHN,3,1773558000
1628,LIN,2,NaN,2026,Chinese Grand Prix,2026-03-15 07:00:00+00:00,Shanghai,CHN,3,1773558000
1629,COL,2,NaN,2026,Chinese Grand Prix,2026-03-15 07:00:00+00:00,Shanghai,CHN,3,1773558000
1630,PER,2,NaN,2026,Chinese Grand Prix,2026-03-15 07:00:00+00:00,Shanghai,CHN,3,1773558000
1631,BOT,2,NaN,2026,Chinese Grand Prix,2026-03-15 07:00:00+00:00,Shanghai,CHN,3,1773558000


In [20]:
driver_price.head()

,driver,race,price,year,race_name,start_date,location,country_code,month,start_epoch
0,VER,1,26.9,2023,Bahrain Grand Prix,2023-03-05 15:00:00+00:00,Sakhir,BRN,3,1678028400
1,HAM,1,23.7,2023,Bahrain Grand Prix,2023-03-05 15:00:00+00:00,Sakhir,BRN,3,1678028400
2,LEC,1,21.2,2023,Bahrain Grand Prix,2023-03-05 15:00:00+00:00,Sakhir,BRN,3,1678028400
3,PER,1,18.0,2023,Bahrain Grand Prix,2023-03-05 15:00:00+00:00,Sakhir,BRN,3,1678028400
4,RUS,1,18.6,2023,Bahrain Grand Prix,2023-03-05 15:00:00+00:00,Sakhir,BRN,3,1678028400


In [21]:
driver_roster.head()

,driver,race,constructor,year
0,VER,1,RBR,2023
1,PER,1,RBR,2023
2,NOR,1,MCL,2023
3,PIA,1,MCL,2023
4,HAM,1,MER,2023


In [22]:
driver_roster.columns

Index(['driver', 'race', 'constructor', 'year'], dtype='object')

In [23]:
driver_price.columns

Index(['driver', 'race', 'price', 'year', 'race_name', 'start_date',
       'location', 'country_code', 'month', 'start_epoch'],
      dtype='object')

In [24]:
driver_points.columns

Index(['driver', 'race', 'points', 'year', 'race_name', 'start_date',
       'location', 'country_code', 'month', 'start_epoch'],
      dtype='object')

In [25]:
#duplicate rows for the merge
driver_price = driver_price.drop(columns = ["race_name", "start_date", "location", "country_code", "month", "start_epoch"])

In [26]:
# two left joins to create the master df
driver_price_points = driver_points.merge(driver_price, how="left", on=["driver", "race", "year"])
driver_price_points = driver_price_points.merge(driver_roster, how="left", on=["year", "driver", "race"])

#renaming
driver_price_points['race_num'] = driver_price_points['race']
driver_price_points = driver_price_points.drop(columns = {"race"})

#some reordering for organizaiton sake
cols = ["year", "race_num", "race_name", "location", "start_date", "driver", "price", "points", "constructor"]
driver_price_points = driver_price_points[cols + [c for c in driver_price_points.columns if c not in cols]]

driver_price_points.tail()

,year,race_num,race_name,location,start_date,driver,price,points,constructor,country_code,month,start_epoch
1723,2026,2,Chinese Grand Prix,Shanghai,2026-03-15 07:00:00+00:00,BOR,7.0,NaN,AUD,CHN,3,1773558000
1724,2026,2,Chinese Grand Prix,Shanghai,2026-03-15 07:00:00+00:00,LIN,6.8,NaN,RB,CHN,3,1773558000
1725,2026,2,Chinese Grand Prix,Shanghai,2026-03-15 07:00:00+00:00,COL,6.4,NaN,ALP,CHN,3,1773558000
1726,2026,2,Chinese Grand Prix,Shanghai,2026-03-15 07:00:00+00:00,PER,5.8,NaN,CAD,CHN,3,1773558000
1727,2026,2,Chinese Grand Prix,Shanghai,2026-03-15 07:00:00+00:00,BOT,5.3,NaN,CAD,CHN,3,1773558000


### Price Based Features
- price_change_previous_race - done
- price_rank - done
- points_per_mill_last_three

In [27]:
driver_price_points = driver_price_points.sort_values(["driver","start_date"])
driver_price_points["price_change_prev_race"] = driver_price_points.groupby("driver")["price"].diff().round(2)
driver_price_points["price_rank"] = driver_price_points.groupby(["year","race_num"])["price"].rank(ascending=False, method="dense")


In [28]:
driver_price_points["year"].value_counts()

year
2024    648
2025    552
2023    484
2026     44
Name: count, dtype: int64

### Points Based Features
- points_last_three_avg - done
- points_last_five_avg - done
-ppm average for the last three races
- ppm average for the last 5 races
-season total ppm



In [29]:
driver_price_points = driver_price_points.sort_values(["driver","year","race_num"]).copy()

driver_price_points["points_last_three_avg"] = (
    driver_price_points.groupby("driver")["points"]
      .transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
      .round(2)
)

driver_price_points["points_last_five_avg"] = (
    driver_price_points.groupby("driver")["points"]
      .transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())
      .round(2)
)


driver_price_points["ppm_last_3"] = (driver_price_points["points_last_three_avg"] / driver_price_points["price"].replace(0, np.nan)).round(2)
driver_price_points["ppm_last_5"] = (driver_price_points["points_last_five_avg"] / driver_price_points["price"].replace(0, np.nan)).round(2)

In [30]:
driver_price_points['year'].value_counts()

year
2024    648
2025    552
2023    484
2026     44
Name: count, dtype: int64

In [31]:
driver_price_points = driver_price_points.sort_values(["year", "race_num", "price"], ascending=[True, True, False]) #back to original sort order
driver_price_points.tail(5)

,year,race_num,race_name,location,start_date,driver,price,points,constructor,country_code,month,start_epoch,price_change_prev_race,price_rank,points_last_three_avg,points_last_five_avg,ppm_last_3,ppm_last_5
1725,2026,2,Chinese Grand Prix,Shanghai,2026-03-15 07:00:00+00:00,COL,6.4,NaN,ALP,CHN,3,1773558000,0.2,18.0,4.00,-0.8,0.62,-0.12
1722,2026,2,Chinese Grand Prix,Shanghai,2026-03-15 07:00:00+00:00,LAW,6.3,NaN,RB,CHN,3,1773558000,-0.2,19.0,2.33,5.8,0.37,0.92
1721,2026,2,Chinese Grand Prix,Shanghai,2026-03-15 07:00:00+00:00,HUL,6.2,NaN,AUD,CHN,3,1773558000,-0.6,20.0,-4.67,0.4,-0.75,0.06
1726,2026,2,Chinese Grand Prix,Shanghai,2026-03-15 07:00:00+00:00,PER,5.8,NaN,CAD,CHN,3,1773558000,-0.2,21.0,-9.67,3.0,-1.67,0.52
1727,2026,2,Chinese Grand Prix,Shanghai,2026-03-15 07:00:00+00:00,BOT,5.3,NaN,CAD,CHN,3,1773558000,-0.6,22.0,-6.67,-4.0,-1.26,-0.75


### Dataset 1 will be used for the training of the model and will be further split into test and training sets

#### Filtering
- making sure that the only rows that exist are those that have points and cost for every record. The ones that do not have a price/points are not active for the given week anyway

In [32]:
hist_driver_price_points = driver_price_points[(~driver_price_points['price'].isna()) & (~driver_price_points['points'].isna())]

#### Exporting dataset 1

In [33]:
file_path = working_directory / "data" / "semi-clean" / "hist_driver_points_df_v1.csv"

hist_driver_price_points.to_csv(file_path)

### Dataset 2 will be used to create predictions for the current weeks race
- this should be held seperate as it has missing values

#### Exporting dataset 2

In [34]:

def get_current_week_race_num():
    today_dt = pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")
    #calendar load
    file_path = working_directory / "data" / "clean" / "race_session_meeting_info.csv"
    calendar = pd.read_csv(file_path).drop(columns=["Unnamed: 0"])

    #getting the current race number
    calendar["start_date"] = pd.to_datetime(calendar["start_date"], utc=True)
    next_race_int = calendar.loc[calendar["start_date"] >= today_dt, "race"].iloc[0].astype(int)

    #getting current year

    today_dt = pd.to_datetime(today_dt)
    year = today_dt.year


    return next_race_int, year



In [35]:
current_week_race_num, current_year = get_current_week_race_num()
print(f'Race: {current_week_race_num}, Year: {current_year}')

Race: 2, Year: 2026


In [36]:
current_week_driver_price_points = driver_price_points[(driver_price_points["race_num"] == current_week_race_num) & (driver_price_points["year"] == current_year)]

In [37]:
current_week_driver_price_points.tail()

,year,race_num,race_name,location,start_date,driver,price,points,constructor,country_code,month,start_epoch,price_change_prev_race,price_rank,points_last_three_avg,points_last_five_avg,ppm_last_3,ppm_last_5
1725,2026,2,Chinese Grand Prix,Shanghai,2026-03-15 07:00:00+00:00,COL,6.4,NaN,ALP,CHN,3,1773558000,0.2,18.0,4.00,-0.8,0.62,-0.12
1722,2026,2,Chinese Grand Prix,Shanghai,2026-03-15 07:00:00+00:00,LAW,6.3,NaN,RB,CHN,3,1773558000,-0.2,19.0,2.33,5.8,0.37,0.92
1721,2026,2,Chinese Grand Prix,Shanghai,2026-03-15 07:00:00+00:00,HUL,6.2,NaN,AUD,CHN,3,1773558000,-0.6,20.0,-4.67,0.4,-0.75,0.06
1726,2026,2,Chinese Grand Prix,Shanghai,2026-03-15 07:00:00+00:00,PER,5.8,NaN,CAD,CHN,3,1773558000,-0.2,21.0,-9.67,3.0,-1.67,0.52
1727,2026,2,Chinese Grand Prix,Shanghai,2026-03-15 07:00:00+00:00,BOT,5.3,NaN,CAD,CHN,3,1773558000,-0.6,22.0,-6.67,-4.0,-1.26,-0.75


#### Exporting Dataset 2

In [38]:
file_path = working_directory / "data" / "semi-clean" / "current_week_driver_points_df_v1.csv"

current_week_driver_price_points.to_csv(file_path)